## 📊 SVOMPTR Dataset Analysis & Cleaning Pipeline
This notebook analyzes the generated synthetic dataset for syntactic diversity (Simple, Compound, Complex), domain distribution (Technical vs Daily), and filters out low-quality translations using Multi-lingual Semantic Similarity.

In [ ]:
# %%capture is used to hide the noisy installation output.
# Removing it temporarily if you want to see installation progress/errors.
import os
print("📥 Installing Analysis Tools...")
get_ipython().system('pip install --quiet datasets sentence-transformers spacy matplotlib seaborn')
get_ipython().system('python -m spacy download en_core_web_sm --quiet')
print("✅ Environment Ready.")

### 1. Load Dataset

In [ ]:
import json
import os
import pandas as pd
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util

DATASET_PATH = "/content/drive/MyDrive/svomptr_auto_train/datasets/synthetic_100k_high_quality.jsonl"
CLEANED_PATH = "/content/drive/MyDrive/svomptr_auto_train/datasets/synthetic_100k_cleaned.jsonl"

if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "./synthetic_100k_high_quality.jsonl" 

data = []
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except: pass

if len(data) > 0:
    df = pd.DataFrame(data)
    print(f"✅ Loaded {len(df)} samples from {DATASET_PATH}.")
else:
    print("❌ No data found to analyze.")
    df = pd.DataFrame()

### 2. Sentence Complexity Analysis

In [ ]:
if not df.empty:
    nlp = spacy.load("en_core_web_sm")

    def classify_sentence(text):
        doc = nlp(text)
        num_verbs = sum(1 for token in doc if token.pos_ == "VERB")
        num_conjs = sum(1 for token in doc if token.pos_ in ["CCONJ", "SCONJ"])
        if num_verbs <= 1 and num_conjs == 0: return "Simple"
        elif num_conjs >= 1 and any(token.dep_ == "mark" or token.dep_ == "advcl" for token in doc): return "Complex"
        elif num_conjs >= 1: return "Compound"
        else: return "Simple"

    sample_df = df.sample(min(5000, len(df)))
    sample_df['complexity'] = sample_df['en'].apply(classify_sentence)
    
    plt.figure(figsize=(8,5))
    sns.countplot(x='complexity', data=sample_df, palette="viridis", hue='complexity', legend=False)
    plt.title("Distribution of Sentence Complexity")
    plt.show()

### 3. Semantic Quality Filtering (LaBSE)

In [ ]:
if not df.empty:
    print("Loading Multi-lingual Embedding Model (LaBSE)...")
    try:
        embedder = SentenceTransformer('sentence-transformers/LaBSE')
        
        print(f"🚀 Scoring {len(df)} rows...")
        en_embeddings = embedder.encode(df['en'].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
        my_embeddings = embedder.encode(df['my'].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
        
        scores = util.cos_sim(en_embeddings, my_embeddings).diagonal().cpu().numpy()
        df['similarity_score'] = scores
        
        cleaned_df = df[df['similarity_score'] >= 0.55]
        print(f"📊 Cleaned: {len(df)} -> {len(cleaned_df)} samples.")
        
        # Export logic
        cleaned_df.drop(columns=['similarity_score']).to_json(CLEANED_PATH, orient='records', lines=True, force_ascii=False)
        print(f"✅ Saved to: {CLEANED_PATH}")
    except Exception as e:
        print(f"❌ Filtering Error: {e}")